In [ ]:
import os
import psycopg2
import keyring

# Define database connection details
DB_NAME = "image_dataset"  # ✅ Set DB_NAME before using it
DB_USER = "postgres"
DB_PASSWORD = keyring.get_password("PostgreSQL", "postgres")
DB_HOST = "localhost"
DB_PORT = "5432"

# Establish connection
try:
    conn = psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT
    )
    cursor = conn.cursor()
    print("✅ PostgreSQL connection established!")

except Exception as e:
    print(f"❌ Error connecting to database: {e}")


In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
import os
import time
import urllib.request

# Path to ChromeDriver
driver_path = r"C:\Program Files\chromedriver-win64\chromedriver.exe"

# Base directory for saving images
base_dir = r"C:\Users\postgres\yolo_project\mined"

# Initialize Selenium WebDriver correctly
service = Service(driver_path)
driver = webdriver.Chrome(service=service)

# Test WebDriver
driver.get("https://www.duckduckgo.com")
print("✅ ChromeDriver is working!")
driver.quit()


In [ ]:
from duckduckgo_search import DDGS
import requests
from PIL import Image
from io import BytesIO

# Function to fetch images using DuckDuckGo API
def fetch_image(category):
    ddg = DDGS()
    results = ddg.images(category, max_results=1)  # Get 1 image per category

    if results:
        image_url = results[0]["image"]
        print(f"🔍 {category}: {image_url}")

        # Fetch and display the image without saving
        response = requests.get(image_url)
        img = Image.open(BytesIO(response.content))
        img.show()

    else:
        print(f"⚠️ No images found for {category}")

# Test with a refined query
fetch_image("construction crane")


In [ ]:
# Loading 10 cranes

from duckduckgo_search import DDGS
import requests
from PIL import Image
from io import BytesIO

# Define categories
categories = ["construction crane", "bulldozer", "forklift", "excavator", "cement mixer",
              "dump truck", "backhoe", "loader", "paver", "hard hat", "safety vest", 
              "safety goggles", "gloves", "boots", "harness", "respirator mask",
              "scaffolding", "barricade", "traffic cone", "construction sign", 
              "wheelbarrow", "ladder", "cables wiring", "unstable structure", 
              "exposed wiring", "falling debris", "fire risk", "oil spill"]

# Function to scrape images for a given category
def scrape_images(category):
    ddg = DDGS()
    results = ddg.images(category, max_results=10)  # Retrieve 10 images
    image_urls = [img["image"] for img in results if "image" in img]

    if image_urls:
        print(f"🔍 {category}: {len(image_urls)} images found")
    else:
        print(f"⚠️ No images found for {category}")

    return image_urls

# Test the function with one category
scrape_images("construction crane")


In [ ]:
# Scraping with thumbnail instead 

import requests

def validate_url(url):
    try:
        response = requests.head(url, allow_redirects=True, timeout=5)
        
        # Check if response is successful and content-type is an image
        if response.status_code == 200 and "image" in response.headers.get("content-type", ""):
            return True
        
        print(f"⚠️ Invalid image URL: {url} (Status {response.status_code})")
        return False
    
    except requests.RequestException as e:
        print(f"❌ URL validation error: {url} - {e}")
        return False

def scrape_images_thumbnail(category):
    ddg = DDGS()
    results = ddg.images(category, max_results=10)
    
    images = []
    for res in results:
        image_url = res.get("image")
        thumbnail_url = res.get("thumbnail")

        # Prioritize full image, fallback to thumbnail if blocked
        images.append(image_url if validate_url(image_url) else thumbnail_url)

    return [img for img in images if img]

# Test the function with one category
scrape_images_thumbnail("construction crane")

In [ ]:
import psycopg2
import urllib.request

# Works to save 10 images at a time
def save_images(category, image_urls, cursor, conn):
    dataset_splits = ["train"] * 5 + ["val"] * 3 + ["test"] * 2

    for idx, url in enumerate(image_urls):
        folder_path = os.path.join(base_dir, dataset_splits[idx], category)
        os.makedirs(folder_path, exist_ok=True)
        image_path = os.path.join(folder_path, f"{category}_{idx}.jpg")

        try:
            # Download Image
            urllib.request.urlretrieve(url, image_path)
            print(f"✅ Saved {image_path}")

            # Insert metadata into PostgreSQL using functions
            cursor.execute("SELECT save_image_metadata(%s, %s, %s)", (url, image_path, "Unknown"))
            image_id = cursor.fetchone()[0]

            # Assign category & partition
            cursor.execute("SELECT save_image_category(%s, %s, %s)", (image_id, category, "low"))
            cursor.execute("SELECT save_dataset_partition(%s, %s, %s)", (image_id, dataset_splits[idx], "Web_scraped"))

            conn.commit()

        except Exception as e:
            print(f"❌ Failed to save {image_path}: {e}")


In [ ]:
base_dir = r"C:\Users\postgres\yolo_project\mined"  # Define storage directory
construction_crane_images = scrape_images_thumbnail("construction crane")
save_images("construction crane", construction_crane_images, cursor, conn)

In [ ]:
base_dir = r"C:\Users\postgres\yolo_project\mined"  # Define storage directory
construction_crane_images = scrape_images("construction crane")
save_images("construction crane", construction_crane_images, cursor, conn)

In [ ]:
# Verify saved images
import os

for split in ["train", "val", "test"]:
    split_path = os.path.join(base_dir, split)
    if os.path.exists(split_path):
        for category in os.listdir(split_path):
            category_path = os.path.join(split_path, category)
            images = os.listdir(category_path) if os.path.isdir(category_path) else []
            print(f"📂 {category} ({split}): {len(images)} images")
    else:
        print(f"⚠️ {split} directory does not exist.")


Mining 100 Images per Category now 

In [ ]:
import os
import time
import requests
from duckduckgo_search import DDGS
from PIL import Image
from io import BytesIO

# Define dataset partitions and their respective splits
split_ratio = {"train": 50, "val": 30, "test": 20}

# Categories to scrape
categories = [
    "construction crane", "bulldozer", "forklift", "excavator", "cement mixer",
    "dump truck", "backhoe", "loader", "paver", "hard hat", "safety vest",
    "safety goggles", "gloves", "boots", "harness", "respirator mask",
    "scaffolding", "barricade", "traffic cone", "construction sign",
    "wheelbarrow", "ladder", "cables wiring", "unstable structure",
    "exposed wiring", "falling debris", "fire risk", "oil spill"
]

# Base directory
base_path = r"E:\yolo_project\mined"
os.makedirs(base_path, exist_ok=True)

def scrape_images(category):
    """Scrapes a single image for a given category."""
    ddg = DDGS()
    try:
        results = ddg.images(category, max_results=1)  # Fetch only one image per request
        if results and "image" in results[0]:
            return results[0]["image"]
    except Exception as e:
        print(f"⚠️ Ratelimit hit while fetching {category}. Skipping...")
    return None

def save_image(img_url, category, split, index):
    """Downloads and saves an image with error handling."""
    try:
        response = requests.get(img_url, timeout=10)
        response.raise_for_status()
        img = Image.open(BytesIO(response.content))

        category_path = os.path.join(base_path, split, category.replace(" ", "_"))
        os.makedirs(category_path, exist_ok=True)

        img_name = f"{category.replace(' ', '_')}_{split}_{index}.jpg"
        img.save(os.path.join(category_path, img_name))

        print(f"✅ Saved: {img_name} in {split}/{category}")
    except Exception as e:
        print(f"❌ Failed to fetch {category} image {index}: {e}")

# Fetch images 100 times, scraping one per category per loop iteration
for i in range(100):
    print(f"🔄 Iteration {i+1} / 100")
    
    for category in categories:
        img_url = scrape_images(category)
        if img_url:
            # Assign dataset split based on ratio distribution
            split = "train" if i < 50 else "val" if i < 80 else "test"
            save_image(img_url, category, split, i+1)

    time.sleep(3)  # Delay between iterations

print("🎯 Mining complete! Images structured correctly in E:\\yolo_project\\mined\\train, val, and test.")


Got rate limited by duck duck go

In [ ]:
import os
import time
import random
import requests
from duckduckgo_search import DDGS
from PIL import Image
from io import BytesIO

# Define dataset partitions and their respective splits
split_ratio = {"train": 50, "val": 30, "test": 20}

# Categories to scrape
categories = [
    "construction crane", "bulldozer", "forklift", "excavator", "cement mixer",
    "dump truck", "backhoe", "loader", "paver", "hard hat", "safety vest",
    "safety goggles", "gloves", "boots", "harness", "respirator mask",
    "scaffolding", "barricade", "traffic cone", "construction sign",
    "wheelbarrow", "ladder", "cables wiring", "unstable structure",
    "exposed wiring", "falling debris", "fire risk", "oil spill"
]

# Base directory
base_path = r"E:\yolo_project\mined"
os.makedirs(base_path, exist_ok=True)

def scrape_images(category):
    """Scrapes a single image for a given category."""
    ddg = DDGS()
    try:
        results = ddg.images(category, max_results=1)  # Fetch only one image per request
        if results and "image" in results[0]:
            return results[0]["image"]
    except Exception as e:
        print(f"⚠️ Ratelimit hit while fetching {category}. Skipping...")
    return None

def save_image(img_url, category, split, index):
    """Downloads and saves an image with error handling."""
    try:
        response = requests.get(img_url, timeout=10)
        response.raise_for_status()
        img = Image.open(BytesIO(response.content))

        category_path = os.path.join(base_path, split, category.replace(" ", "_"))
        os.makedirs(category_path, exist_ok=True)

        img_name = f"{category.replace(' ', '_')}_{split}_{index}.jpg"
        img.save(os.path.join(category_path, img_name))

        print(f"✅ Saved: {img_name} in {split}/{category}")
    except Exception as e:
        print(f"❌ Failed to fetch {category} image {index}: {e}")

# Fetch images sequentially with varying delay between requests
for i in range(100):
    print(f"🔄 Iteration {i+1} / 100")

    for category in categories:
        img_url = scrape_images(category)
        if img_url:
            split = "train" if i < 50 else "val" if i < 80 else "test"
            save_image(img_url, category, split, i+1)

        # Introduce varying sleep time to prevent rate limiting
        delay = random.uniform(30, 60)  # Random delay between 5-20 seconds
        print(f"⏳ Waiting {delay:.2f} seconds before next request...")
        time.sleep(delay)

print("🎯 Mining complete! Images structured correctly in E:\\yolo_project\\mined\\train, val, and test.")


Trying Pexels instead

In [2]:
import requests

PEXELS_API_KEY = "OzA5NAaBNk7otiMnUDvQS4ZwodwYraXZHHzw5FDV05WxGZTO6T1UmJN3"
url = "https://api.pexels.com/v1/search?query=construction"

headers = {"Authorization": PEXELS_API_KEY}
response = requests.get(url, headers=headers)

# Extract rate limit headers
rate_limit = response.headers.get("X-Ratelimit-Limit")
remaining = response.headers.get("X-Ratelimit-Remaining")
reset_time = response.headers.get("X-Ratelimit-Reset")

print(f"Rate Limit: {rate_limit}, Remaining: {remaining}, Reset Time: {reset_time}")


Rate Limit: 25000, Remaining: 24937, Reset Time: 1752376841


In [1]:
import os
import time
import random
import requests
from pexelsapi.pexels import Pexels


from PIL import Image
from io import BytesIO

# 🔑 Set up Pexels API client
pexel = Pexels("OzA5NAaBNk7otiMnUDvQS4ZwodwYraXZHHzw5FDV05WxGZTO6T1UmJN3")


# ✅ Define dataset partitions (50-30-20 split)
split_ratio = {"train": 50, "val": 30, "test": 20}

# ✅ Define categories to mine
categories = [
    "construction crane", "bulldozer", "forklift", "excavator", "cement mixer",
    "dump truck", "backhoe", "loader", "paver", "hard hat", "safety vest",
    "safety goggles", "gloves", "boots", "harness", "respirator mask",
    "scaffolding", "barricade", "traffic cone", "construction sign",
    "wheelbarrow", "ladder", "cables wiring", "unstable structure",
    "exposed wiring", "falling debris", "fire risk", "oil spill"
]

# 📂 Set base directory for dataset
base_path = r"E:\yolo_project\mined"
os.makedirs(base_path, exist_ok=True)

# 🔄 Fetch images for each category
def fetch_image(category):
    """Fetches one image per category from Pexels."""
    try:
        photos = pexel.search_photos(query=category, per_page=1)
        if photos and "photos" in photos:
            return photos["photos"][0]["src"]["original"]  # ✅ Correct path for image URL
    except Exception as e:
        print(f"⚠️ API Error while fetching {category}: {e}")
    return None

# 💾 Download and save image
def save_image(img_url, category, split, index):
    """Downloads and stores images with proper error handling."""
    try:
        response = requests.get(img_url, timeout=10)
        response.raise_for_status()
        img = Image.open(BytesIO(response.content))

        # ✅ Ensure category folder uses underscores
        category_folder = category.replace(" ", "_")
        category_path = os.path.join(base_path, split, category_folder)
        os.makedirs(category_path, exist_ok=True)

        # ✅ Ensure image name matches category transformation
        img_name = f"{category_folder}_{split}_{index}.jpg"
        img.save(os.path.join(category_path, img_name))

        print(f"✅ Saved: {img_name} in {split}/{category_folder}")
    except Exception as e:
        print(f"❌ Failed to fetch {category} image {index}: {e}")

# ⏳ Sequential mining with varying delay to prevent API rate limits
for i in range(100):
    print(f"🔄 Iteration {i+1} / 100")

    for category in categories:
        img_url = fetch_image(category)
        if img_url:
            split = "train" if i < 50 else "val" if i < 80 else "test"
            save_image(img_url, category, split, i+1)

        # ⏳ Introduce random sleep time between requests
        delay = random.uniform(5, 6)  # ✅ Increased random delay to prevent rate limits
        print(f"⏳ Waiting {delay:.2f} seconds before next request...")
        time.sleep(delay)

print("🎯 Mining complete! Images structured correctly in E:\\yolo_project\\mined\\train, val, and test.")


🔄 Iteration 1 / 100
✅ Saved: construction_crane_train_1.jpg in train/construction_crane
⏳ Waiting 5.65 seconds before next request...
✅ Saved: bulldozer_train_1.jpg in train/bulldozer
⏳ Waiting 5.41 seconds before next request...
✅ Saved: forklift_train_1.jpg in train/forklift
⏳ Waiting 5.08 seconds before next request...
✅ Saved: excavator_train_1.jpg in train/excavator
⏳ Waiting 5.66 seconds before next request...
✅ Saved: cement_mixer_train_1.jpg in train/cement_mixer
⏳ Waiting 5.16 seconds before next request...
✅ Saved: dump_truck_train_1.jpg in train/dump_truck
⏳ Waiting 5.54 seconds before next request...
✅ Saved: backhoe_train_1.jpg in train/backhoe
⏳ Waiting 5.64 seconds before next request...
✅ Saved: loader_train_1.jpg in train/loader
⏳ Waiting 5.18 seconds before next request...
✅ Saved: paver_train_1.jpg in train/paver
⏳ Waiting 5.61 seconds before next request...
✅ Saved: hard_hat_train_1.jpg in train/hard_hat
⏳ Waiting 5.42 seconds before next request...
✅ Saved: safety_

KeyboardInterrupt: 

Code above mines the same image again and again

In [3]:
import os
import time
import random
import requests
from pexelsapi.pexels import Pexels


from PIL import Image
from io import BytesIO

# 🔑 Set up Pexels API client
pexel = Pexels("OzA5NAaBNk7otiMnUDvQS4ZwodwYraXZHHzw5FDV05WxGZTO6T1UmJN3")


# ✅ Define dataset partitions (50-30-20 split)
split_ratio = {"train": 50, "val": 30, "test": 20}

# ✅ Define categories to mine
categories = [
    "construction crane", "bulldozer", "forklift", "excavator", "cement mixer",
    "dump truck", "backhoe", "loader", "paver", "hard hat", "safety vest",
    "safety goggles", "gloves", "boots", "harness", "respirator mask",
    "scaffolding", "barricade", "traffic cone", "construction sign",
    "wheelbarrow", "ladder", "cables wiring", "unstable structure",
    "exposed wiring", "falling debris", "fire risk", "oil spill"
]

# 📂 Set base directory for dataset
base_path = r"E:\yolo_project\mined"
os.makedirs(base_path, exist_ok=True)

# 🔄 Fetch images for each category
def fetch_image(category, page):
    """Fetches a different image per category by randomizing page numbers."""
    try:
        # Request a random page each time
        photos = pexel.search_photos(query=category, page=page, per_page=1)

        if photos and "photos" in photos:
            return photos["photos"][0]["src"]["original"]  # ✅ Get new image from different page
    except Exception as e:
        print(f"⚠️ API Error while fetching {category}: {e}")
    return None

# 💾 Download and save image
def save_image(img_url, category, split, index):
    """Downloads and stores images with proper error handling."""
    try:
        response = requests.get(img_url, timeout=10)
        response.raise_for_status()
        img = Image.open(BytesIO(response.content))

        # ✅ Ensure category folder uses underscores
        category_folder = category.replace(" ", "_")
        category_path = os.path.join(base_path, split, category_folder)
        os.makedirs(category_path, exist_ok=True)

        # ✅ Ensure image name matches category transformation
        img_name = f"{category_folder}_{split}_{index}.jpg"
        img.save(os.path.join(category_path, img_name))

        print(f"✅ Saved: {img_name} in {split}/{category_folder}")
    except Exception as e:
        print(f"❌ Failed to fetch {category} image {index}: {e}")

# ⏳ Sequential mining with varying delay to prevent API rate limits
for i in range(100):
    print(f"🔄 Iteration {i+1} / 100")

    for category in categories:
        random_page = random.randint(1, 20)  # ✅ Request different pages for new images
        img_url = fetch_image(category, random_page)

        if img_url:
            split = "train" if i < 50 else "val" if i < 80 else "test"
            save_image(img_url, category, split, i+1)

        delay = random.uniform(5, 6)
        print(f"⏳ Waiting {delay:.2f} seconds before next request...")
        time.sleep(delay)

print("🎯 Mining complete! Images structured correctly in E:\\yolo_project\\mined\\train, val, and test.")


🔄 Iteration 1 / 100
✅ Saved: construction_crane_train_1.jpg in train/construction_crane
⏳ Waiting 5.18 seconds before next request...
✅ Saved: bulldozer_train_1.jpg in train/bulldozer
⏳ Waiting 5.75 seconds before next request...
✅ Saved: forklift_train_1.jpg in train/forklift
⏳ Waiting 5.05 seconds before next request...
✅ Saved: excavator_train_1.jpg in train/excavator
⏳ Waiting 5.10 seconds before next request...
✅ Saved: cement_mixer_train_1.jpg in train/cement_mixer
⏳ Waiting 5.28 seconds before next request...
✅ Saved: dump_truck_train_1.jpg in train/dump_truck
⏳ Waiting 5.32 seconds before next request...
✅ Saved: backhoe_train_1.jpg in train/backhoe
⏳ Waiting 5.08 seconds before next request...
✅ Saved: loader_train_1.jpg in train/loader
⏳ Waiting 5.66 seconds before next request...
✅ Saved: paver_train_1.jpg in train/paver
⏳ Waiting 5.84 seconds before next request...
✅ Saved: hard_hat_train_1.jpg in train/hard_hat
⏳ Waiting 5.40 seconds before next request...
✅ Saved: safety_

Automatic rate tracking and cooldown handling

In [ ]:
import os
import time
import random
import requests
from pexelsapi.pexels import Pexels
from PIL import Image
from io import BytesIO

# 🔑 Pexels API setup
PEXELS_API_KEY = "YOUR_API_KEY"
pexel = Pexels(PEXELS_API_KEY)

# ✅ Dataset partitions (50-30-20 split)
split_ratio = {"train": 50, "val": 30, "test": 20}

# ✅ Categories to mine
categories = [
    "construction crane", "bulldozer", "forklift", "excavator", "cement mixer",
    "dump truck", "backhoe", "loader", "paver", "hard hat", "safety vest",
    "safety goggles", "gloves", "boots", "harness", "respirator mask",
    "scaffolding", "barricade", "traffic cone", "construction sign",
    "wheelbarrow", "ladder", "cables wiring", "unstable structure",
    "exposed wiring", "falling debris", "fire risk", "oil spill"
]

# 📂 Base directory for dataset
base_path = r"E:\yolo_project\mined"
os.makedirs(base_path, exist_ok=True)

# 🔄 Fetch images for each category
def fetch_image(category):
    """Fetch one image per category from Pexels."""
    try:
        photos = pexel.search_photos(query=category, per_page=1)
        if photos and "photos" in photos:
            return photos["photos"][0]["src"]["original"]
    except Exception as e:
        print(f"⚠️ API Error while fetching {category}: {e}")
    return None

# 🛑 Check API Rate Limits
def check_rate_limit():
    """Fetch API headers and manage cooldown."""
    url = "https://api.pexels.com/v1/search?query=test"
    headers = {"Authorization": PEXELS_API_KEY}
    response = requests.get(url, headers=headers)
    
    limit = int(response.headers.get("X-Ratelimit-Limit", 25000))  # Default: 25K
    remaining = int(response.headers.get("X-Ratelimit-Remaining", 0))
    reset_time = int(response.headers.get("X-Ratelimit-Reset", time.time()))

    print(f"📊 Rate Limit: {limit}, Remaining: {remaining}, Reset Time: {reset_time}")

    if remaining < 50:
        wait_time = max(0, reset_time - int(time.time()))
        print(f"⏳ Nearing limit! Pausing for {wait_time / 60:.2f} minutes...")
        time.sleep(wait_time)  # Wait until reset

# 💾 Save image locally
def save_image(img_url, category, split, index):
    """Downloads and stores images with error handling."""
    try:
        response = requests.get(img_url, timeout=10)
        response.raise_for_status()
        img = Image.open(BytesIO(response.content))

        category_folder = category.replace(" ", "_")
        category_path = os.path.join(base_path, split, category_folder)
        os.makedirs(category_path, exist_ok=True)

        img_name = f"{category_folder}_{split}_{index}.jpg"
        img.save(os.path.join(category_path, img_name))

        print(f"✅ Saved: {img_name} in {split}/{category_folder}")
    except Exception as e:
        print(f"❌ Failed to fetch {category} image {index}: {e}")

# ⏳ Sequential mining with API cooldown handling
for i in range(100):
    print(f"🔄 Iteration {i+1} / 100")

    for category in categories:
        check_rate_limit()  # ✅ Fetch rate limit before requesting images

        img_url = fetch_image(category)
        if img_url:
            split = "train" if i < 50 else "val" if i < 80 else "test"
            save_image(img_url, category, split, i+1)

        # ⏳ Introduce random sleep time between requests
        delay = random.uniform(5, 30)  # ✅ Increased random delay
        print(f"⏳ Waiting {delay:.2f} seconds before next request...")
        time.sleep(delay)

print("🎯 Mining complete! Images structured correctly in E:\\yolo_project\\mined\\train, val, and test.")
